# Question 1.1: Loading Data + Cleaning

The `research_abstracts.json` file was loaded into a pandas DataFrame to inspect its structure before preprocessing. It contains 1,138 records and three columns: `DOI`, `Abstract` and `Label`. The abstract is the text that will be cleaned for analysis, while the label identifies whether each paper belongs to the `ai` or `classical` category.

The initial inspection found no missing abstracts or labels, although four records have no DOI. Since the DOI is a paper identifier rather than a description of its content, it will be retained for reference but excluded from the text features used by the classifier. The labels are also unevenly distributed, so this will be considered when the data is divided for model evaluation.

In [2]:
import json
from pathlib import Path
import pandas as pd

dataset_path = Path("research_abstracts.json")

with dataset_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

Research_Abstracts = pd.DataFrame(data)

print("Dataset shape:", Research_Abstracts.shape)
print("Dataset columns:", Research_Abstracts.columns)
print("Missing values in each column:\n", Research_Abstracts.isnull().sum())
print("Label counts:\n", Research_Abstracts["Label"].value_counts())

Dataset shape: (1138, 3)
Dataset columns: Index(['DOI', 'Abstract', 'Label'], dtype='object')
Missing values in each column:
 DOI         4
Abstract    0
Label       0
dtype: int64
Label counts:
 Label
classical    900
ai           238
Name: count, dtype: int64


### Checking for duplicate abstracts

Before applying text-cleaning steps, duplicate abstracts are checked. This helps identify repeated records and abstracts that may have been assigned different labels. The DOI is not required for classifying abstract content, so missing DOI values do not prevent the records from being used.

In [3]:
duplicate_pairs = Research_Abstracts.duplicated(subset=['Abstract', 'Label']).sum()

conflicting_abstracts = Research_Abstracts.groupby('Abstract')['Label'].nunique().gt(1).sum()

print("Repeated abstract and label pairs:", duplicate_pairs)
print("Abstracts associated with more than one label:", conflicting_abstracts)

Repeated abstract and label pairs: 13
Abstracts associated with more than one label: 0


### Text denoising and duplicate removal

The dataset contains 13 repeated abstract–label pairs and no abstracts assigned to more than one label. The repeated records are removed to prevent the same abstract from being counted more than once. The DOI is excluded from the modelling dataset because it identifies a paper but does not describe its content.

The abstract text is cleaned by converting it to lowercase, removing HTML tags and web links, replacing punctuation with spaces, and standardising whitespace. The original abstract is kept alongside the cleaned version so the changes can be checked. English stop words will be handled during TF-IDF feature extraction.

In [4]:
import html 
import re

# Keep only the text and target columns, DOI is metadata, not a text feature 
Research_Abstracts_Cleaned = Research_Abstracts[['Abstract', 'Label']].copy()

# Remove rows without text or a target label 
Research_Abstracts_Cleaned = Research_Abstracts_Cleaned.dropna(subset=['Abstract', 'Label'])

# Standardise spacing and label columns 
Research_Abstracts_Cleaned['Abstract'] = Research_Abstracts_Cleaned['Abstract'].astype(str).str.strip()
Research_Abstracts_Cleaned['Label'] = Research_Abstracts_Cleaned['Label'].astype(str).str.strip().str.lower()

# Remove repeated abstract label pairs 
rows_before_duplicates = len(Research_Abstracts_Cleaned)
Research_Abstracts_Cleaned = Research_Abstracts_Cleaned.drop_duplicates(subset=['Abstract', 'Label'])
duplicates_removed = rows_before_duplicates - len(Research_Abstracts_Cleaned)

# Clean the abstract text 
def clean_abstract(text):
    text = html.unescape(text)  # Decode HTML entities
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)  # remove web links
    text = text.lower()
    text = re.sub(r"[^\w\s-]", " ", text)  # remove punctuation, keep hyphens
    text = re.sub(r"\s+", " ", text).strip()  # standardise whitespace
    return text

Research_Abstracts_Cleaned['Cleaned_Abstract'] = Research_Abstracts_Cleaned['Abstract'].apply(clean_abstract)

print("Rows after cleaning:", len(Research_Abstracts_Cleaned))
print("Repeated records removed:", duplicates_removed)
print("Class counts after cleaning:\n", Research_Abstracts_Cleaned['Label'].value_counts())
print("Example of cleaned text:", Research_Abstracts_Cleaned['Cleaned_Abstract'].iloc[0][:500]) # Print first 500 characters of the first cleaned abstract
 

Rows after cleaning: 1125
Repeated records removed: 13
Class counts after cleaning:
 Label
classical    890
ai           235
Name: count, dtype: int64
Example of cleaned text: forward-looking forecasting of the inflation rate could help the central bank and other government departments to better use monetary policy to stabilize prices and prevent the impact of inflation on market entities especially for low- and middle-income groups it can also help financial institutions and investors better make investment decisions in this sense the forecast of inflation rate is of great significance the existing literature mainly uses linear models such as autoregressive ar and ve


### Previewing the research abstracts

The first five records are displayed to inspect the DOI, abstract text and category label before preprocessing.

In [5]:
from IPython.display import display

pd.set_option("display.max_colwidth", 300)

display(Research_Abstracts[["DOI", "Abstract", "Label"]].head(5))

,DOI,Abstract,Label
0,10.1155/2021/1071145,"Forward-looking forecasting of the inflation rate could help the central bank and other government departments to better use monetary policy to stabilize prices and prevent the impact of inflation on market entities, especially for low- and middle-income groups. It can also help financial instit...",ai
1,10.1007/s43546-022-00384-2,"Our study examined the disaggregation of inflation components in Nigeria using the stacked ensemble approach, a machine learning algorithm capable of compensating the weakness of an ensemble and a base learner with the strength of another. This approach gives flexibility of a synergistic perform...",ai
2,10.1002/for.2667,This paper develops a dynamic factor model that uses euro area country-specific information on output and inflation to estimate an area-wide measure of the output gap. Our model assumes that output and inflation can be decomposed into country-specific stochastic trends and a common cyclical comp...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model to examine Japan’s soft lockdown policies in 2021 under the COVID-19 crisis. As real-time research, this paper consists of two parts written during two different research periods. The first part, which was originally reported in February 2021, studie...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifying conditions under which individual stakeholders in a collective action problem interact in ways that are more cooperative and in the best interest of the collective. The literature applying game theory to healthcare markets predicts that when provid...,classical


# Question 1.2: Feature Engineering + Modelling 

The `Cleaned_Abstract` column is selected as the predictor because it contains the text used to classify each research paper. The `Label` column is selected as the target because it contains the category the model must predict (`ai` or `classical`). The `DOI` column is excluded because it is an identifier, rather than a meaningful description of the abstract’s content.

TF-IDF is selected to represent the abstracts numerically. Unigrams capture individual words, while bigrams capture pairs of consecutive words that may carry useful context. The vectoriser will be fitted on the training data during modelling to avoid using information from the test data when building the vocabulary (scikit-learn developers, n.d.).

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_text = Research_Abstracts_Cleaned['Cleaned_Abstract']
y = Research_Abstracts_Cleaned['Label']

tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df = 2)

print("Text input column: Cleaned_Abstract")
print("Target column: Label")
print("Number of records:", len(X_text))
print("Target categories:", sorted(y.unique()))
print("TF-IDF features: unigrams and bigrams")

Text input column: Cleaned_Abstract
Target column: Label
Number of records: 1125
Target categories: ['ai', 'classical']
TF-IDF features: unigrams and bigrams


# Question 1.3: Build Baseline Model 

A Logistic Regression classifier is used as the baseline model to predict whether an abstract is labelled `ai` or `classical`. The data is divided into training and testing sets using an 80/20 stratified split. Stratification helps preserve the original proportions of the two labels in each set.

TF-IDF feature extraction and classification are combined in a pipeline. This means the vocabulary is learned from the training data and then applied to the test data, reducing the risk of information leakage. Balanced class weights are used to account for the smaller number of `ai` abstracts. Performance is assessed on both the training set and the held-out test set to compare model learning with generalisation.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Divide the data into training and testing sets using an 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size = 0.20, random_state = 42, stratify = y)

# Build a pipeline so TF-IDF is only fitted on the training data 
Baseline_Model = Pipeline([
    ("tfidf", tfidf_vectorizer),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    ))
])

# Train the baseline model 
Baseline_Model.fit(X_train, y_train)

# Make predictions on the training and testing sets
train_predictions = Baseline_Model.predict(X_train)
test_predictions = Baseline_Model.predict(X_test)

# Evaluate the model's performance and test performance 
print("Training Accuracy:", round(accuracy_score(y_train, train_predictions),3))
print("Testing Accuracy:", round(accuracy_score(y_test, test_predictions),3))

print("Training Classification Report:\n", classification_report(y_train, train_predictions, digits = 3))
print("Testing Classification Report:\n", classification_report(y_test, test_predictions, digits = 3))

# Show test-set predictions in a confusion matrix
matrix = confusion_matrix(y_test, test_predictions, labels=["ai", "classical"])
confusion_df = pd.DataFrame(matrix, index=["Actual ai", "Actual classical"], columns=["Predicted ai", "Predicted classical"])
print("Confusion Matrix:\n", confusion_df)

Training Accuracy: 0.982
Testing Accuracy: 0.898
Training Classification Report:
               precision    recall  f1-score   support

          ai      0.926     0.995     0.959       188
   classical      0.999     0.979     0.989       712

    accuracy                          0.982       900
   macro avg      0.962     0.987     0.974       900
weighted avg      0.983     0.982     0.982       900

Testing Classification Report:
               precision    recall  f1-score   support

          ai      0.750     0.766     0.758        47
   classical      0.938     0.933     0.935       178

    accuracy                          0.898       225
   macro avg      0.844     0.849     0.847       225
weighted avg      0.899     0.898     0.898       225

Confusion Matrix:
                   Predicted ai  Predicted classical
Actual ai                   36                   11
Actual classical            12                  166


### Baseline model performance and generalisation

The Logistic Regression model achieved 98.2% accuracy on the training data and 89.8% accuracy on the held-out test data. The lower test score indicates that the model performed better on the data used for learning than on unseen data. The 8.4 percentage-point difference suggests some overfitting, although the test results indicate that the model can still classify many abstracts correctly.

Performance differed between the two categories. For `classical` abstracts, the model achieved an F1-score of 0.935 and correctly classified 166 of the 178 test examples. For `ai` abstracts, it achieved an F1-score of 0.758 and correctly identified 36 of the 47 examples, misclassifying 11 as `classical`. The macro-average F1-score was 0.847, which gives equal weight to both categories. This is useful because the dataset contains substantially more `classical` abstracts than `ai` abstracts.

Overall, the model provides a useful baseline, but its lower performance on the `ai` category shows that the results should be interpreted with care. The model’s test-set performance measures how well it generalised to this particular split of the dataset; it does not establish how it would perform on every collection of research papers.

# Question 2.1: Four Machine Learning Models

Four classifiers are trained using the same training and test sets as the Logistic Regression baseline. Linear SVM and Ridge Classifier provide alternative linear approaches, while Random Forest and Extra Trees use tree-based ensembles. Each model uses the same TF-IDF settings so that the results can be compared fairly.

Accuracy and macro F1-score are reported. Macro F1 is included because it gives equal importance to the `ai` and `classical` categories, despite their different numbers of records.

In [8]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Keep the fitted models, so we can inspect and keep best one for later 
trained_models = {"Baseline Logisitic Regression": Baseline_Model}

# Define four additional classifiers 
candidate_classifiers = {"Linear SVM": LinearSVC(class_weight="balanced",random_state=42),
                  "Ridge Classifier": RidgeClassifier(class_weight="balanced"),
                  "Random Forest": RandomForestClassifier(n_estimators = 200,class_weight="balanced",random_state=42, n_jobs=-1),
                  "Extra Trees": ExtraTreesClassifier(n_estimators = 300, class_weight="balanced",random_state=42, n_jobs=-1)
                  
                 }
# Train each clasifier in a TF - IDF pipeline 
for model_name, classifier in candidate_classifiers.items():
    model_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)),
        ("classifier", classifier)
    ])
    
    model_pipeline.fit(X_train, y_train)
    
    # Store the trained model for later inspection 
    trained_models[model_name] = model_pipeline

# Comparing training and testing performance 
results = []

for model_name, fitted_model in trained_models.items():
    train_predictions = fitted_model.predict(X_train)
    test_predictions = fitted_model.predict(X_test) 
    
    results.append({
        "Model": model_name,
        "Train Accuracy": accuracy_score(y_train, train_predictions),
        "Test Accuracy": accuracy_score(y_test, test_predictions),
        "Test macro F1": f1_score(y_test, test_predictions, average='macro')
    })
    
Model_Comparison = pd.DataFrame(results).sort_values(by="Test macro F1", ascending=False)   

display(Model_Comparison.round(3))


,Model,Train Accuracy,Test Accuracy,Test macro F1
4,Extra Trees,1.000,0.933,0.887
1,Linear SVM,0.999,0.916,0.865
3,Random Forest,1.000,0.920,0.860
2,Ridge Classifier,0.999,0.902,0.850
0,Baseline Logisitic Regression,0.982,0.898,0.847


The four additional classifiers achieved higher test macro F1-scores than the Logistic Regression baseline. Extra Trees performed best on this test split, with a test accuracy of 0.933 and macro F1-score of 0.887, compared with 0.898 and 0.847 for the baseline. Ridge Classifier produced only a small macro F1 improvement, while Linear SVM and Random Forest also performed better than the baseline.

Extra Trees achieved 100% training accuracy, compared with 93.3% test accuracy. This difference suggests that it may fit the training data more closely than unseen data. Its test results are the strongest among the models evaluated so far, but the training–test gap should be considered when interpreting its performance.

# Question 2.2 Additional Predictive Performance Enhancement 

The Extra Trees model achieved the highest test macro F1-score in the initial model comparison, so it was selected for further tuning. GridSearchCV tests a defined set of hyperparameter combinations using stratified cross-validation on the training data. Macro F1 is used to select the configuration because it gives equal importance to the `ai` and `classical` categories.

The test set is kept separate during tuning. The tuned model is evaluated on it after the search to determine whether the selected configuration improves on the original Extra Trees model.

In [9]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Create an Extra Trees pipeline for tuning 
ExtraTrees_Pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)),
    ("classifier", ExtraTreesClassifier(class_weight="balanced", random_state=42, n_jobs=-1))
])

# Define the settings to compare 
parameter_grid = {
    "classifier__n_estimators": [200, 300],
    "classifier__max_features": ["sqrt", "log2"],
    "classifier__min_samples_leaf": [1, 2]
}   

# Use stratified 3 fold cross validation on the training data 
cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

Grid_Search = GridSearchCV(
    estimator=ExtraTrees_Pipeline,
    param_grid=parameter_grid,
    scoring="f1_macro",
    cv=cross_validation,
    n_jobs=-1,
    refit = True 
)

# Fit the search using training data only 
Grid_Search.fit(X_train, y_train)

# Save the best tuned pipeline and evaluate it on the held-out test set
Tuned_ExtraTrees_Model = Grid_Search.best_estimator_
tuned_predictions = Tuned_ExtraTrees_Model.predict(X_test)  

original_predictions = trained_models["Extra Trees"].predict(X_test)

print("Best Parameters:", Grid_Search.best_params_)
print("Best cross validation macro F1:", round(Grid_Search.best_score_, 3))

print("Original Extra Trees Test accuracy:", round(accuracy_score(y_test, original_predictions), 3))
print("Original Extra Trees Test macro F1:", round(f1_score(y_test, original_predictions, average='macro'), 3))

print("Tuned Extra Trees Test accuracy:", round(accuracy_score(y_test, tuned_predictions), 3))
print("Tuned Extra Trees Test macro F1:", round(f1_score(y_test, tuned_predictions, average='macro'), 3))

Best Parameters: {'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__n_estimators': 300}
Best cross validation macro F1: 0.836
Original Extra Trees Test accuracy: 0.933
Original Extra Trees Test macro F1: 0.887
Tuned Extra Trees Test accuracy: 0.938
Tuned Extra Trees Test macro F1: 0.896


### Effect of hyperparameter tuning

GridSearchCV selected the Extra Trees configuration with `max_features='sqrt'` and `min_samples_leaf=2`. The tuned model achieved 0.938 test accuracy and a 0.896 test macro F1-score, compared with 0.933 and 0.887 for the original Extra Trees model. This represents a modest improvement on the held-out test set. The best cross-validation macro F1-score was 0.836, and the test set was kept separate during parameter selection.

# Question 2.3: Web Interface 

In [10]:
import joblib

joblib.dump(
    Tuned_ExtraTrees_Model,
    "best_research_abstract_classifier.joblib"
)

print("Tuned model saved successfully.")

Tuned model saved successfully.


In [11]:
Demo_Abstracts = (
    Research_Abstracts_Cleaned
    .groupby("Label", group_keys=False)
    .sample(n=1, random_state=42)[["Label", "Abstract"]]
)

for _, row in Demo_Abstracts.iterrows():
    print(f"Actual category: {row['Label']}")
    print(row["Abstract"])
    print("\n" + "-" * 80 + "\n")

Actual category: ai
Bitcoin has grown in popularity and has now attracted the attention of individual and institutional investors. Accurate Bitcoin price direction forecasts are important for determining the trend in Bitcoin prices and asset allocation. This paper addresses several unanswered questions. How important are business cycle variables like interest rates, inflation, and market volatility for forecasting Bitcoin prices? Does the importance of these variables change across time? Are the most important macroeconomic variables for forecasting Bitcoin prices the same as those for gold prices? To answer these questions, we utilize tree-based machine learning classifiers, along with traditional logit econometric models. The analysis reveals several important findings. First, random forests predict Bitcoin and gold price directions with a higher degree of accuracy than logit models. Prediction accuracy for bagging and random forests is between 75% and 80% for a five-day prediction. 

### Web interface deployment

A Streamlit web interface was created for the tuned research abstract classifier. The deployed dashboard can be accessed here:

 **[https://itnpa4-34researchly-npa-projectcristinaabrahamse-knnahyuwg7hfx.streamlit.app/]()**.

In [12]:
import sklearn
import joblib
import streamlit

print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("streamlit:", streamlit.__version__)

scikit-learn: 1.7.2
joblib: 1.5.2
streamlit: 1.51.0
